# 04 - Colour-magnitude diagrams

Builds a colour-magnitude diagram for every LSST band pair, showing:

* a dense **SN Ia** background plus four **core-collapse contaminant** populations
  (Ibc, IIL, IIn, IIP), each drawn twice - unlensed (black-edged) and lensed;
* the **logistic decision boundary** separating lensed from unlensed events;
* lensed events sampled through **real LSST cadence realisations**, coloured by the
  rest-frame epoch at which the colour method could identify them.

All populations are resampled with rate weights, so what is plotted is a fair draw
from the expected observed population rather than from the raw simulation.

Uses `cmsne.supernovae`, `cmsne.colour_magnitude` and `cmsne.lsst`.

In [ ]:
# If running in Colab, install the dependencies (uncomment):
# !pip install sncosmo
# !pip install git+https://github.com/LSSTDESC/OpSimSummaryV2.git

# Make the `cmsne` package importable when this notebook lives in notebooks/.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from cmsne.lsst import band_pairs, contaminant_info
from cmsne.supernovae import Supernovae, Supernovae2, Nugent
from cmsne.colour_magnitude import (full_creation, combine_populations,
                                    exponential_regression, success_rate)

## Sample sizes

Sized for an overnight run: roughly **4.2 hours** on measured per-event costs of
4.5 ms (cadence path), 15.5 ms (SN Ia background) and 14.1 ms (per contaminant
class), across 10 band pairs and 4 contaminant classes.

Cadence-path acceptance is **7-14%**. It was ~35% higher before observations were
phased against a real `t0`: supernovae exploding in a cadence gap, or too late in
the survey, are now genuinely missed rather than having their time axis shifted so
the first visit became first light. The per-event cost roughly halved at the same
time, because the visibility window is far narrower than the old three-year cut.

Redshift is **not** capped globally. Each generator defaults to
`redshift_limits(...)`, the exact range over which its model covers that band pair,
so the redder pairs keep their full reach (`lsstz-lssty` runs to z = 3.01 where
`lsstg-lsstr` stops at 0.93) while nothing is wasted above the limit. That change
alone lifted cadence-path acceptance from ~2%, and took the two fast generators
to 100%.

Memory: the fast generators hold a 1000-point light curve per event, so
`N_BACKGROUND` costs ~62 KB/event and each contaminant class ~31 KB/event - about
1.7 GB live per band pair, released between pairs.

In [ ]:
N_BACKGROUND  = 10000   # SN Ia events per band pair (model-only, fast)
N_CONTAMINANT = 8000    # events per contaminant class (model-only, fast)
N_CADENCE     = 200000  # events through a real OpSim cadence (slow, the scarce one)
N_DRAW        = 600     # rate-weighted draws per population, for the fit and the plot
PLOT_MAX_CADENCE = 2500 # cap on cadence points *drawn* per panel (stats use all)

Generator  = Supernovae()    # cadence realisations, incl. delayed second image
Generator2 = Supernovae2()   # SN Ia background
Generator3 = Nugent()        # core-collapse contaminants

## Generate the populations

One pass per band pair. Each band pair is `(bluer, redder)`; the redder band is the
magnitude axis and the colour is `bluer - redder`.

In [ ]:
import time
from cmsne.supernovae import redshift_limits

plot_data_per_band = []

# Collected across all band pairs, for the shared colourbar and for notebook 05.
all_l_times, all_time_delays, all_timdel = [], [], []
all_redshifts, all_magnifications, all_weights_l = [], [], []

run_start = time.time()

for pair_index, pair in enumerate(band_pairs, start=1):
    pair_start = time.time()
    blue_band, red_band = pair
    band1, band2 = red_band, blue_band   # band1 = magnitude axis, band2 = colour's blue side

    z_lo, z_hi = redshift_limits('salt3', band1, band2)
    print(f"[{pair_index}/{len(band_pairs)}] {blue_band}-{red_band}  "
          f"SN Ia z range {z_lo:.2f}-{z_hi:.2f}", flush=True)

    # --- fast model-only populations: SN Ia plus contaminants ----------------
    SNe_Ia = Generator2.generate_many(N_BACKGROUND, band1, band2)
    contaminants = [
        Generator3.generate_many(N_CONTAMINANT, band1, band2, abs_mag, sigma, source)
        for abs_mag, sigma, source in contaminant_info
    ]

    # Rate-weighted draws for each population, plotted separately.
    drawn = {'Ia': full_creation(SNe_Ia, N_DRAW)}
    for (_, _, source), pop in zip(contaminant_info, contaminants):
        drawn[source] = full_creation(pop, N_DRAW)

    # Decision boundary fitted on all populations pooled together.
    all_colour_l, all_colour_ul, all_peak_mag_l, all_peak_mag_ul = combine_populations(
        [SNe_Ia] + contaminants, N_DRAW)
    x_range, y_range, m_, b_ = exponential_regression(
        all_peak_mag_ul, all_peak_mag_l, all_colour_ul, all_colour_l)

    # --- slow path: events observed through a real LSST cadence -------------
    SNe_data = Generator.generate_many(N_CADENCE, band1, band2)

    l_peak_mags, l_colors, l_times = [], [], []
    for sne_event in SNe_data:
        l_peak_mags.append(sne_event['band_1_mag_l'])
        l_colors.append(sne_event['lensed_colour'])
        l_times.append(sne_event['time_of_SN'])

        all_l_times.append(sne_event['time_of_SN'])
        all_time_delays.append(sne_event['time_delay'])
        all_timdel.append(sne_event['timdel'])          # physical lensing delay
        all_redshifts.append(sne_event['z'])
        all_magnifications.append(sne_event['magnification'])
        all_weights_l.append(sne_event['weights_l'])

    # Each event is compared against the boundary at its OWN peak magnitude.
    rate = success_rate(l_peak_mags, l_colors, m_, b_)

    plot_data_per_band.append({
        'band_pair': pair, 'drawn': drawn,
        'x_range': x_range, 'y_range': y_range,
        'l_peak_mags': l_peak_mags, 'l_colors': l_colors, 'l_times': l_times,
        'success_rate': rate, 'n_cadence_events': len(SNe_data),
    })

    elapsed = time.time() - pair_start
    eta = (time.time() - run_start) / pair_index * (len(band_pairs) - pair_index)
    print(f"      {len(SNe_data)} cadence events kept of {N_CADENCE}"
          f" ({100 * len(SNe_data) / N_CADENCE:.1f}%), success rate = {rate:.3f}"
          f"  [{elapsed / 60:.1f} min, ETA {eta / 60:.0f} min]", flush=True)

print(f"\ntotal: {(time.time() - run_start) / 3600:.2f} h")

## Plot the grid

In [ ]:
STYLES = {
    'Ia':            ('tab:gray', 'SN Ia'),
    'nugent-sn1bc':  ('b',        'SN Ibc'),
    'nugent-sn2l':   ('r',        'SN IIL'),
    'nugent-sn2n':   ('c',        'SN IIn'),
    'nugent-sn2p':   ('g',        'SN IIP'),
}

# Shared colour scale for the epoch colouring.
valid_l_times = np.array([t for t in all_l_times if np.isfinite(t)])
if valid_l_times.size:
    vmin_time, vmax_time = valid_l_times.min(), valid_l_times.max()
else:
    vmin_time, vmax_time = 0.0, 1.0
    print("Warning: no lensed SNe with valid epochs; colourbar scale is arbitrary.")

ncols = 4
nrows = (len(plot_data_per_band) + ncols - 1) // ncols
# constrained_layout reserves room for the two-line titles and the colourbar.
# The previous subplots_adjust + fig.add_axes combination left no vertical gap
# between rows, so each title overlapped the axis label of the panel above.
fig, ax = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.4 * nrows),
                       constrained_layout=True)
ax_flat = np.atleast_1d(ax).flatten()

# Drop unused panels before the colourbar is attached to the survivors.
for k in range(len(plot_data_per_band), nrows * ncols):
    fig.delaxes(ax_flat[k])
used_axes = list(ax_flat[:len(plot_data_per_band)])

for n, data in enumerate(plot_data_per_band):
    current_ax = ax_flat[n]
    current_ax.plot(data['x_range'], data['y_range'], color='black', lw=1.5)

    for key, (colour, label) in STYLES.items():
        mag_ul, colour_ul, _, mag_l, colour_l, _ = data['drawn'][key]
        # Unlensed: black-edged. Lensed: plain fill, and the one that gets a label.
        current_ax.scatter(mag_ul, colour_ul, c=colour, edgecolors='black',
                           s=14, linewidths=.4, alpha=.75)
        current_ax.scatter(mag_l, colour_l, c=colour, label=label, s=14, alpha=.75)

    # Statistics use every cadence event; the overlay is subsampled so a panel with
    # tens of thousands of them stays readable rather than saturating to a blob.
    pm = np.asarray(data['l_peak_mags'], dtype=float)
    cl = np.asarray(data['l_colors'], dtype=float)
    tm = np.asarray(data['l_times'], dtype=float)
    if pm.size > PLOT_MAX_CADENCE:
        sel = np.random.choice(pm.size, PLOT_MAX_CADENCE, replace=False)
        pm, cl, tm = pm[sel], cl[sel], tm[sel]

    scat = current_ax.scatter(pm, cl, c=tm, cmap='spring',
                              vmin=vmin_time, vmax=vmax_time, s=9, alpha=.6,
                              linewidths=0)

    blue_band, red_band = data['band_pair']
    current_ax.set_title(f"{blue_band} vs {red_band}\n"
                         f"success rate = {data['success_rate']:.2f}", fontsize=11)
    current_ax.set_xlabel(f"{red_band} peak magnitude")
    current_ax.set_ylabel(f"{blue_band} - {red_band}")
    if n == 0:
        current_ax.legend(loc='upper left', fontsize=8)

norm = plt.Normalize(vmin=vmin_time, vmax=vmax_time)
sm = cm.ScalarMappable(cmap='spring', norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=used_axes, fraction=0.02, pad=0.015)
cbar.set_label('Rest-frame epoch of identification (days from peak)')

plt.show()

## Success rate per band pair

Fraction of cadence-realisation lensed events that fall above the decision
boundary, i.e. that the colour method would flag.

In [ ]:
for data in plot_data_per_band:
    blue_band, red_band = data['band_pair']
    print(f"{blue_band:>6} - {red_band:<6}  "
          f"success rate = {data['success_rate']:.3f}  "
          f"(n = {data['n_cadence_events']})")

## Save the cadence-realisation sample for notebook 05

The time-delay diagnostics reuse this population rather than regenerating it.

In [ ]:
np.savez('cadence_population.npz',
         l_times=np.array(all_l_times, dtype=float),
         time_delays=np.array(all_time_delays, dtype=float),
         timdel=np.array(all_timdel, dtype=float),
         redshifts=np.array(all_redshifts, dtype=float),
         magnifications=np.array(all_magnifications, dtype=float),
         weights_l=np.array(all_weights_l, dtype=float))
print(f"saved {len(all_time_delays)} events to cadence_population.npz")